In [1]:
import json
import os
import networkx as nx
from plan import PartialPlan
from old.run_mzn import run_mzn
from old.mzn_arr_to_schedule import *
from objects import Service, Movement

In [2]:
walking_distances = {
    ("entry", "52"):3,
    ("entry", "53"):4,
    ("entry", "54"):5,
    ("entry", "55"):6,
    ("entry", "56"):7,
    ("entry", "57"):8,
    ("entry", "58"):9,
    ("entry", "59"):10,
    ("entry", "60"):8,
    ("entry", "61_service"):9,
    ("entry", "62_service"):10,
    ("entry", "63"):12,
    ("52", "53"):1,
    ("52", "54"):2,
    ("52", "55"):3,
    ("52", "56"):4,
    ("52", "57"):5,
    ("52", "58"):6,
    ("52", "59"):7,
    ("52", "60"):5,
    ("52", "61_service"):6,
    ("52", "62_service"):7,
    ("52", "63"):10,
    ("53", "54"):1,
    ("53", "55"):2,
    ("53", "56"):3,
    ("53", "57"):4,
    ("53", "58"):5,
    ("53", "59"):6,
    ("53", "60"):4,
    ("53", "61_service"):5,
    ("53", "62_service"):6,
    ("53", "63"):9,
    ("54", "55"):1,
    ("54", "56"):2,
    ("54", "57"):3,
    ("54", "58"):4,
    ("54", "59"):5,
    ("54", "60"):3,
    ("54", "61_service"):4,
    ("54", "62_service"):5,
    ("54", "63"):8,
    ("55", "56"):1,
    ("55", "57"):2,
    ("55", "58"):3,
    ("55", "59"):4,
    ("55", "60"):4,
    ("55", "61_service"):3,
    ("55", "62_service"):4,
    ("55", "63"):7,
    ("56", "57"):1,
    ("56", "58"):2,
    ("56", "59"):3,
    ("56", "60"):5,
    ("56", "61_service"):4,
    ("56", "62_service"):3,
    ("56", "63"):6,
    ("57", "58"):1,
    ("57", "59"):2,
    ("57", "60"):6,
    ("57", "61_service"):5,
    ("57", "62_service"):4,
    ("57", "63"):7,
    ("58", "59"):1,
    ("58", "60"):7,
    ("58", "61_service"):6,
    ("58", "62_service"):5,
    ("58", "63"):8,
    ("59", "60"):8,
    ("59", "61_service"):7,
    ("59", "62_service"):6,
    ("59", "63"):9,
    ("60", "61_service"):1,
    ("60", "62_service"):2,
    ("60", "63"):4,
    ("61_service", "62_service"):1,
    ("61_service", "63"):3,
    ("62_service", "63"):4,
}

In [3]:
rows = list()
for cfg in range(1, 50):

    for num_t in range(3, 25):
        plan_file = f"../results/tops/base4/pln_{cfg}_{num_t}t.txt"
        if not os.path.exists(plan_file):
            # rows.append({'config':cfg,'num_trains':num_t,'solved':False,'num_expansions':None,'makespan':None})
            continue
        with open(plan_file, 'r') as f:
            lines = f.readlines()
        # plan_idxs = [i for i in range(len(lines)) if lines[i].lower().startswith('found new plan')]
        cost_idxs = [i for i in range(len(lines)) if lines[i].lower().startswith('cost')]
        
        
        if len(cost_idxs) < 1:
            print(f'No makespans for num_t {num_t}!')
            cost = None
            solved = False
        else:
            cost = int(float(lines[cost_idxs[-1]].split(':')[1].strip()))
            solved = True
        
        plan_idxs = [i for i in range(len(lines)) if lines[i].lower().startswith('found plan')]
        sol_idxs = [i for i in range(len(lines)) if lines[i].lower().startswith('expansions')]
        if len(plan_idxs) < 1 or len(sol_idxs) < 1:
            print(f'No plan for num_t {num_t}!')
            cost = None
            solved = False
        else:
            plan_lines = lines[plan_idxs[-1]+1:sol_idxs[-1]]


        exp_idxs = [i for i in range(len(lines)) if lines[i].lower().startswith('expansions')]
        if len(exp_idxs) < 1:
            print(f'No expansions for num_t {num_t}!')
            exp = -1
        else:
            exp = int(float(lines[exp_idxs[-1]].split(':')[1].split('state')[0].strip()))


        if solved:
            plan_lines = [f"({line.replace(',', '').replace('(',' ').replace(')','').replace('\n','')})" for line in plan_lines if 'move' in line or 'service' in line]
            print(plan_lines)
            pp = PartialPlan(plan_lines)
            pp.build_constraints()
            pp.build_walking_times_matrix(walking_distances)
            pp.write_dzn(1)
            
            try:
                [start_times, durations, action_driver, action_train] = run_mzn(300, 'chuffed')
            except:
                print(f"error? - {num_t}")
                continue

            train_schedule, driver_schedule = init_train_driver_schedules(start_times,durations, 
                                                                            action_train, action_driver)
            driver_schedule = finalize_driver_schedule(driver_schedule)
            movement_labels = []
            for a in pp.actions:
                if type(a) is Service:
                    movement_labels.append(f'service {a.train.name.split("_")[1]} {a.track.name.split("_")[1]}')
                else:
                    movement_labels.append(f'move {a.train.name.split("_")[1]} {a.origin.name.split("_")[1]} {a.destination.name.split("_")[1]}')

            pp_dur = max([start_times[i]+durations[i] for i in range(len(start_times))])

            plot_schedule(train_schedule, ['idle']+movement_labels, f'../results/tops/base4/plots/plot_{num_t}t')

            rows.append({'config':cfg,'num_trains':num_t,'num_expansions':exp,'cost':cost,'makespan_pp':int(pp_dur)})


df = pd.DataFrame(rows)
if not os.path.exists('results_base4_tops.csv'):
    df.to_csv('results_base4_tops.csv', index=False)
else:
    df_old = pd.read_csv('results_base4_tops.csv')
    df_new = pd.concat([df,df_old], ignore_index=True)
    df_new = df_new.drop_duplicates(subset=['config','num_trains'])
    df_new.to_csv('results_base4_tops.csv', index=False)

['(move_bside_onto_empty train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train train_slt40 track_61_service)', '(move_bside_onto_empty train_slt41 track_entry track_56)', '(move_aside_onto_empty train_slt40 track_61_service track_59)', '(move_bside_onto_empty train_slt42 track_entry track_58)', '(move_bside_onto_empty train_slt41 track_56 track_62_service)', '(service_train train_slt41 track_62_service)', '(move_aside_onto_occupied train_slt41 track_62_service track_59)', '(move_bside_onto_empty train_slt42 track_58 track_61_service)', '(service_train train_slt42 track_61_service)', '(move_aside_onto_occupied train_slt42 track_61_service track_59)']
{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 54, "flatIntVars": 12, "flatBoolConstraints": 27, "flatIntConstraints": 94, "evaluatedReifiedConstraints": 54, "method": "minimize", "flatTime": 0.298714}}
{"type": "statistics", "statistics": {"nSolutions": 2}}


['(move_bside_onto_empty train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train train_slt40 track_61_service)', '(move_bside_onto_empty train_slt41 track_entry track_56)', '(move_bside_onto_empty train_slt42 track_entry track_59)', '(move_bside_onto_empty train_slt43 track_entry track_58)', '(move_aside_onto_empty train_slt40 track_61_service track_57)', '(move_bside_onto_occupied train_slt44 track_entry track_56)', '(move_bside_onto_empty train_slt41 track_56 track_62_service)', '(service_train train_slt41 track_62_service)', '(move_bside_onto_empty train_slt42 track_59 track_61_service)', '(service_train train_slt42 track_61_service)', '(move_bside_onto_occupied train_slt45 track_entry track_58)', '(move_aside_onto_empty train_slt41 track_62_service track_59)', '(move_bside_onto_empty train_slt44 track_56 track_62_service)', '(service_train train_slt44 track_62_service)', '(move_aside_onto_occupied train_slt42 track_61_ser

['(move_bside_onto_empty train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train train_slt40 track_61_service)', '(move_bside_onto_empty train_slt41 track_entry track_56)', '(move_bside_onto_empty train_slt42 track_entry track_59)', '(move_bside_onto_empty train_slt43 track_entry track_58)', '(move_bside_onto_empty train_slt44 track_entry track_57)', '(move_bside_onto_occupied train_slt45 track_entry track_56)', '(move_bside_onto_occupied train_slt46 track_entry track_59)', '(move_bside_onto_empty train_slt41 track_56 track_62_service)', '(service_train train_slt41 track_62_service)', '(move_bside_onto_occupied train_slt47 track_entry track_59)', '(move_bside_onto_occupied train_slt48 track_entry track_58)', '(move_bside_onto_empty train_slt45 track_56 track_60)', '(move_bside_onto_empty train_slt45 track_60 track_63)', '(move_aside_onto_empty train_slt40 track_61_service track_56)', '(move_bside_onto_empty train_slt42 track_

['(move_bside_onto_empty train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train train_slt40 track_61_service)', '(move_bside_onto_empty train_slt41 track_entry track_56)', '(move_bside_onto_empty train_slt42 track_entry track_59)', '(move_bside_onto_empty train_slt43 track_entry track_58)', '(move_bside_onto_empty train_slt44 track_entry track_57)', '(move_bside_onto_occupied train_slt45 track_entry track_56)', '(move_bside_onto_occupied train_slt46 track_entry track_59)', '(move_bside_onto_occupied train_slt47 track_entry track_59)', '(move_bside_onto_occupied train_slt48 track_entry track_58)', '(move_bside_onto_empty train_slt41 track_56 track_62_service)', '(service_train train_slt41 track_62_service)', '(move_bside_onto_occupied train_slt49 track_entry track_57)', '(move_bside_onto_empty train_slt50 track_entry track_54)', '(move_bside_onto_empty train_slt51 track_entry track_52)', '(move_bside_onto_empty train_slt51 tr

{"type": "statistics", "statistics": {"nSolutions": 3}}
['(move_bside_onto_empty train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train train_slt40 track_61_service)', '(move_bside_onto_empty train_slt41 track_entry track_56)', '(move_bside_onto_empty train_slt42 track_entry track_59)', '(move_bside_onto_empty train_slt43 track_entry track_58)', '(move_bside_onto_empty train_slt44 track_entry track_57)', '(move_bside_onto_occupied train_slt45 track_entry track_56)', '(move_bside_onto_occupied train_slt46 track_entry track_59)', '(move_bside_onto_occupied train_slt47 track_entry track_59)', '(move_bside_onto_occupied train_slt48 track_entry track_58)', '(move_bside_onto_occupied train_slt49 track_entry track_57)', '(move_bside_onto_empty train_slt41 track_56 track_62_service)', '(service_train train_slt41 track_62_service)', '(move_aside_onto_occupied train_slt40 track_61_service track_56)', '(move_bside_onto_empty train_slt4

...

['(enter_bside_and_move_onto_empty driver_andy train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train driver_andy train_slt40 track_61_service)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_entry track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_entry track_59)', '(enter_bside_and_move_onto_empty driver_andy train_slt43 track_entry track_58)', '(enter_bside_and_move_onto_empty driver_andy train_slt44 track_entry track_57)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_56 track_62_service)', '(service_train driver_andy train_slt41 track_62_service)', '(enter_aside_and_move_onto_empty driver_andy train_slt40 track_61_service track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_59 track_61_service)', '(service_train driver_andy train_slt42 track_61_service)', '(enter_bside_and_move_onto_empty driver_andy train_slt45 track_entry track_59)', '

['(enter_bside_and_move_onto_empty driver_andy train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train driver_andy train_slt40 track_61_service)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_entry track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_entry track_59)', '(enter_bside_and_move_onto_empty driver_andy train_slt43 track_entry track_58)', '(enter_bside_and_move_onto_empty driver_andy train_slt44 track_entry track_57)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_56 track_62_service)', '(service_train driver_andy train_slt41 track_62_service)', '(enter_aside_and_move_onto_empty driver_andy train_slt40 track_61_service track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_59 track_61_service)', '(service_train driver_andy train_slt42 track_61_service)', '(enter_bside_and_move_onto_empty driver_andy train_slt45 track_entry track_59)', '

['(enter_bside_and_move_onto_empty driver_andy train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train driver_andy train_slt40 track_61_service)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_entry track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_entry track_59)', '(enter_bside_and_move_onto_empty driver_andy train_slt43 track_entry track_58)', '(enter_bside_and_move_onto_empty driver_andy train_slt44 track_entry track_57)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_56 track_62_service)', '(service_train driver_andy train_slt41 track_62_service)', '(enter_aside_and_move_onto_empty driver_andy train_slt40 track_61_service track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_59 track_61_service)', '(service_train driver_andy train_slt42 track_61_service)', '(enter_bside_and_move_onto_empty driver_andy train_slt45 track_entry track_59)', '

{"type": "statistics", "statistics": {"nSolutions": 1}}
['(enter_bside_and_move_onto_empty driver_andy train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train driver_andy train_slt40 track_61_service)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_entry track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_entry track_59)', '(enter_bside_and_move_onto_empty driver_andy train_slt43 track_entry track_58)', '(enter_bside_and_move_onto_empty driver_andy train_slt44 track_entry track_57)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_56 track_62_service)', '(service_train driver_andy train_slt41 track_62_service)', '(enter_aside_and_move_onto_empty driver_andy train_slt40 track_61_service track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_59 track_61_service)', '(service_train driver_andy train_slt42 track_61_service)', '(enter_bside_and_move_onto

['(enter_bside_and_move_onto_empty driver_andy train_slt40 track_entry track_56)', '(move_bside_onto_empty train_slt40 track_56 track_61_service)', '(service_train driver_andy train_slt40 track_61_service)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_entry track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_entry track_59)', '(enter_bside_and_move_onto_empty driver_andy train_slt43 track_entry track_58)', '(enter_bside_and_move_onto_empty driver_andy train_slt44 track_entry track_57)', '(enter_bside_and_move_onto_empty driver_andy train_slt41 track_56 track_62_service)', '(service_train driver_andy train_slt41 track_62_service)', '(enter_aside_and_move_onto_empty driver_andy train_slt40 track_61_service track_56)', '(enter_bside_and_move_onto_empty driver_andy train_slt42 track_59 track_61_service)', '(service_train driver_andy train_slt42 track_61_service)', '(enter_bside_and_move_onto_empty driver_andy train_slt45 track_entry track_59)', '

In [4]:
# cfg_list = list(range(7,50))
# cfg_list.reverse()
# for cfg in cfg_list:
#     for num_t in range(3,16):
#         plan_file = f"../results/enhsp/base3/pln_{cfg}_{num_t}t.txt"
#         if not os.path.exists(plan_file):
#             # rows.append({'config':cfg,'num_trains':num_t,'solved':False,'num_expansions':None,'makespan':None})
#             print('not exists')
#             continue
#         os.rename(plan_file, f"../results/enhsp/base3/pln_{cfg+1}_{num_t}t.txt")